# Applied AI Engineer – Movie Plot RAG System

## Objective

Build a lightweight Retrieval-Augmented Generation (RAG) system that answers movie plot questions using a movie plot dataset.

The system uses semantic retrieval with FAISS, cross-encoder reranking, and Gemini for grounded answer generation.

## 1. Install Dependencies

The required Python libraries are installed directly in the Google Colab runtime.

In [ ]:
!pip -q install datasets

## 2. Imports & Configuration

Import the libraries required for data processing, semantic embeddings, FAISS retrieval, cross-encoder reranking, and Gemini-based answer generation.

## 3. Load Dataset

Load the movie plot dataset and inspect its structure, columns, and sample records before preprocessing.

In [ ]:
!wget -q -O movie_plots.csv \
"https://raw.githubusercontent.com/PacktPublishing/Elastic-Stack-8.x-Cookbook/main/Chapter2/dataset/wiki_movie_plots_deduped.csv"

import pandas as pd

df = pd.read_csv("movie_plots.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (34886, 8)

Columns:
['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast', 'Genre', 'Wiki Page', 'Plot']


## 4. Data Preprocessing & Chunking

Select the required movie plot fields, clean the text, and split movie plots into smaller chunks for semantic retrieval.

In [ ]:
# Keep only the fields required by the assessment
movies = df[["Title", "Plot"]].copy()

# Remove missing values
movies = movies.dropna(subset=["Title", "Plot"])

# Remove empty plot/title values
movies = movies[
    (movies["Title"].str.strip() != "") &
    (movies["Plot"].str.strip() != "")
]

# Select a reproducible subset of 300 movies
movies = movies.sample(n=300, random_state=42).reset_index(drop=True)

print("Selected movies:", len(movies))
print("\nMissing values:")
print(movies.isnull().sum())

print("\nSample:")
display(movies.head(5))

Selected movies: 300

Missing values:
Title    0
Plot     0
dtype: int64

Sample:


,Title,Plot
0,The Day the Earth Stood Still,"When a flying saucer lands in Washington, D.C...."
1,The Burning,"One night at Camp Blackfoot, several campers p..."
2,Nobel Chor,"The first Asian Nobel Laureate, Rabindranath T..."
3,Trent's Last Case,A major international financier is found dead ...
4,Aafat,Inspector Amar and Inspector Chhaya are after ...


In [ ]:
def chunk_text(text, chunk_size=300):
    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


# Create chunks for each movie
chunk_records = []

for _, row in movies.iterrows():
    chunks = chunk_text(row["Plot"], chunk_size=300)

    for chunk_id, chunk in enumerate(chunks):
        chunk_records.append({
            "title": row["Title"],
            "chunk_id": chunk_id,
            "text": chunk
        })


chunks_df = pd.DataFrame(chunk_records)

print("Movies:", movies.shape[0])
print("Total chunks:", len(chunks_df))

print("\nAverage words per chunk:",
      round(chunks_df["text"].str.split().str.len().mean(), 2))

print("\nSample chunk:")
display(chunks_df.head(3))

Movies: 300
Total chunks: 558

Average words per chunk: 209.88

Sample chunk:


,title,chunk_id,text
0,The Day the Earth Stood Still,0,"When a flying saucer lands in Washington, D.C...."
1,The Day the Earth Stood Still,1,greatest living person is; Bobby suggests Prof...
2,The Day the Earth Stood Still,2,"the spaceship, then leaves to retrieve Klaatu'..."


In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [ ]:
# Generate embeddings for all movie plot chunks

texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape)

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Number of embeddings: 558
Embedding dimensions: (558, 384)


## 5. Embeddings & FAISS Vector Index

Generate dense vector embeddings for the movie plot chunks and build a FAISS vector index to enable efficient semantic similarity search.

In [ ]:
!pip -q install faiss-cpu

In [ ]:
import faiss
import numpy as np

# Convert embeddings to float32 for FAISS
embedding_matrix = np.asarray(embeddings, dtype="float32")

# Create FAISS index using cosine similarity
# Cosine similarity = inner product on normalized vectors
faiss.normalize_L2(embedding_matrix)

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatIP(dimension)

# Add all embeddings to the index
index.add(embedding_matrix)

print("FAISS index created successfully.")
print("Number of vectors:", index.ntotal)
print("Vector dimension:", dimension)

FAISS index created successfully.
Number of vectors: 558
Vector dimension: 384


In [ ]:
def retrieve(query, top_k=3):
    # Convert the query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Search the FAISS index
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "title": chunks_df.iloc[idx]["title"],
            "chunk_id": int(chunks_df.iloc[idx]["chunk_id"]),
            "text": chunks_df.iloc[idx]["text"],
            "score": float(score)
        })

    return results

In [ ]:
query = "Which movie features an artificial intelligence system?"

results = retrieve(query, top_k=3)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Title:", result["title"])
    print("Chunk ID:", result["chunk_id"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"][:500], "...")


--- Result 1 ---
Title: Josettante Hero (ജോസേട്ടന്റെ ഹീറോ)
Chunk ID: 0
Score: 0.3172
Text: The film tells a story with movie-making as the backdrop. Vijayaraghavan appears as the title character of Josettan, who is a senior film producer. Anoop Menon comes up as Saajan Malyath, the still photographer of the film who is accidentally selected to become the hero of the new film in production. ...

--- Result 2 ---
Title: Cool...Sakkath Hot Maga
Chunk ID: 0
Score: 0.3085
Text: The film has supposedly a comedy and romance mixed storyline. Ganesh plays a college going student in the film. The film has been shot in some picturesque locations such as Egypt, Dubai and Jordan and Middle East.[2] ...

--- Result 3 ---
Title: Margam
Chunk ID: 0
Score: 0.2851
Text: The film tells the story of a revolutionary who, years later, realizes that his efforts have been wasted and witnesses the ill-fate of his co-rebels and leads a secluded life in a state of clinical depression.[5] ...


In [ ]:
# Check whether our selected 300 movies contain AI-related plots

ai_keywords = [
    "artificial intelligence",
    "artificially intelligent",
    "AI system",
    "computer",
    "robot",
    "android",
    "cyborg",
    "machine intelligence"
]

mask = movies["Plot"].str.contains(
    "|".join(ai_keywords),
    case=False,
    na=False
)

ai_movies = movies[mask][["Title", "Plot"]].copy()

print("AI-related movies found:", len(ai_movies))

display(ai_movies[["Title"]].head(20))

AI-related movies found: 9


,Title
0,The Day the Earth Stood Still
5,Zardoz
34,After Hours
73,Watchers
125,Borat! Cultural Learnings of America for Make ...
141,Kickin' It Old Skool
228,Yatterman
238,Predators
294,Rings


In [ ]:
for _, row in ai_movies.iterrows():
    print("\n" + "=" * 80)
    print("TITLE:", row["Title"])
    print("PLOT:")
    print(row["Plot"][:1000])


TITLE: The Day the Earth Stood Still
PLOT:
When a flying saucer lands in Washington, D.C., the Army quickly surrounds it. A humanoid (Michael Rennie) emerges, announcing that he has come in peace. When he unexpectedly opens a small device, he is shot by a nervous soldier. A tall robot emerges from the saucer and quickly disintegrates the soldiers' weapons. The alien orders the robot, Gort, to stop. He explains that the now-broken device was a gift for the President which would have enabled him "to study life on the other planets".
The alien, Klaatu, is taken to Walter Reed Hospital. After surgery, he uses a salve to quickly heal his wound. Meanwhile, the Army is unable to enter the saucer; Gort stands outside, silent and unmoving.
Klaatu tells the President's secretary, Mr. Harley (Frank Conroy), that he has a message that must be delivered to all the world's leaders simultaneously. Harley tells him that such a meeting in the current political climate is impossible. Klaatu suggests th

In [ ]:
query = "Which movie features the robot Gort?"

results = retrieve(query, top_k=5)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Title:", result["title"])
    print("Chunk ID:", result["chunk_id"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"][:700], "...")


--- Result 1 ---
Title: The Day the Earth Stood Still
Chunk ID: 2
Score: 0.3943
Text: the spaceship, then leaves to retrieve Klaatu's body. Gort brings Klaatu back to life, but he explains to Helen that his revival is only temporary. Klaatu addresses Barnhardt's assembled scientists, informing them that he represents an interplanetary organization that created a police force of invincible robots like Gort. "In matters of aggression, we have given them absolute power over us". Klaatu concludes, "Your choice is simple: join us and live in peace, or pursue your present course and face obliteration". Klaatu and Gort re-enter the spaceship and depart. ...

--- Result 2 ---
Title: The Day the Earth Stood Still
Chunk ID: 0
Score: 0.3865
Text: When a flying saucer lands in Washington, D.C., the Army quickly surrounds it. A humanoid (Michael Rennie) emerges, announcing that he has come in peace. When he unexpectedly opens a small device, he is shot by a nervous soldier. A tall robot emerges fr

## 7. RAG Pipeline & Gemini Generation

Combine semantic retrieval, cross-encoder reranking, and Gemini generation into a grounded Retrieval-Augmented Generation pipeline.

In [ ]:
!pip -q install google-genai

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Reply with exactly: Gemini connection successful."
)

print(response.text)

Gemini connection successful.


In [ ]:
def generate_answer(query, results):
    context = "\n\n".join(
        [
            f"Movie: {result['title']}\n"
            f"Plot context: {result['text']}"
            for result in results
        ]
    )

    prompt = f"""
You are a movie plot question-answering assistant.

Answer the user's question using ONLY the retrieved movie plot context below.

If the context does not contain enough information to answer the question,
say that the information is not available in the retrieved context.

User question:
{query}

Retrieved context:
{context}

Give a concise, factual answer.
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

In [ ]:
query = "Which movie features the robot Gort?"

results = retrieve(query, top_k=3)

answer = generate_answer(query, results)

print("Question:", query)
print("\nAnswer:", answer)

Question: Which movie features the robot Gort?

Answer: Based on the provided context, the robot Gort is featured in the movie ***The Day the Earth Stood Still***.


In [ ]:
import json

def build_json_output(query, results, answer):
    contexts = [
        result["text"]
        for result in results
    ]

    reasoning = (
        f"The question was answered by retrieving the most relevant "
        f"movie plot chunks using semantic similarity and using those "
        f"retrieved contexts to generate the answer."
    )

    output = {
        "answer": answer,
        "contexts": contexts,
        "reasoning": reasoning
    }

    return output

In [ ]:
output = build_json_output(query, results, answer)

print(json.dumps(output, indent=2, ensure_ascii=False))

{
  "answer": "Based on the provided context, the robot Gort is featured in the movie ***The Day the Earth Stood Still***.",
  "contexts": [
    "the spaceship, then leaves to retrieve Klaatu's body. Gort brings Klaatu back to life, but he explains to Helen that his revival is only temporary. Klaatu addresses Barnhardt's assembled scientists, informing them that he represents an interplanetary organization that created a police force of invincible robots like Gort. \"In matters of aggression, we have given them absolute power over us\". Klaatu concludes, \"Your choice is simple: join us and live in peace, or pursue your present course and face obliteration\". Klaatu and Gort re-enter the spaceship and depart.",
    "When a flying saucer lands in Washington, D.C., the Army quickly surrounds it. A humanoid (Michael Rennie) emerges, announcing that he has come in peace. When he unexpectedly opens a small device, he is shot by a nervous soldier. A tall robot emerges from the saucer and qui

In [ ]:
!pip -q install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

retrieval_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("BGE retrieval model loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE retrieval model loaded successfully.


In [ ]:
# Generate BGE embeddings for all movie plot chunks

bge_embeddings = retrieval_model.encode(
    chunks_df["text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Number of embeddings:", len(bge_embeddings))
print("Embedding dimensions:", bge_embeddings.shape)

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Number of embeddings: 558
Embedding dimensions: (558, 384)


In [ ]:
import faiss
import numpy as np

bge_matrix = np.asarray(bge_embeddings, dtype="float32")

bge_dimension = bge_matrix.shape[1]

bge_index = faiss.IndexFlatIP(bge_dimension)
bge_index.add(bge_matrix)

print("BGE FAISS index created successfully.")
print("Number of vectors:", bge_index.ntotal)
print("Vector dimension:", bge_dimension)

BGE FAISS index created successfully.
Number of vectors: 558
Vector dimension: 384


In [ ]:
query = "Which movie features the robot Gort?"

# -----------------------------
# MiniLM retrieval
# -----------------------------
old_query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(old_query_embedding)

old_scores, old_indices = index.search(old_query_embedding, 5)


# -----------------------------
# BGE retrieval
# -----------------------------
bge_query_embedding = retrieval_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

bge_scores, bge_indices = bge_index.search(
    bge_query_embedding,
    5
)


# -----------------------------
# Display comparison
# -----------------------------
print("========== MiniLM ==========")

for rank, (score, idx) in enumerate(
    zip(old_scores[0], old_indices[0]), 1
):
    print(
        f"{rank}. {chunks_df.iloc[idx]['title']} "
        f"(score={score:.4f})"
    )


print("\n========== BGE ==========")

for rank, (score, idx) in enumerate(
    zip(bge_scores[0], bge_indices[0]), 1
):
    print(
        f"{rank}. {chunks_df.iloc[idx]['title']} "
        f"(score={score:.4f})"
    )

========== MiniLM ==========
1. The Day the Earth Stood Still (score=0.3943)
2. The Day the Earth Stood Still (score=0.3865)
3. Yatterman (score=0.3414)
4. Kaizoku Sentai Gokaiger vs. Space Sheriff Gavan: The Movie (score=0.3388)
5. Margam (score=0.3300)

========== BGE ==========
1. The Day the Earth Stood Still (score=0.5972)
2. Kaizoku Sentai Gokaiger vs. Space Sheriff Gavan: The Movie (score=0.5763)
3. The Day the Earth Stood Still (score=0.5588)
4. The Cyclops (score=0.5578)
5.  Kickin' It Old Skool (score=0.5553)


In [ ]:
def retrieve_diverse(query, top_k=3, search_k=10):
    # Create query embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Retrieve more candidates first
    scores, indices = index.search(query_embedding, search_k)

    results = []
    seen_titles = set()

    for score, idx in zip(scores[0], indices[0]):
        title = chunks_df.iloc[idx]["title"]

        # Keep only the best chunk from each movie
        if title in seen_titles:
            continue

        results.append({
            "title": title,
            "chunk_id": int(chunks_df.iloc[idx]["chunk_id"]),
            "text": chunks_df.iloc[idx]["text"],
            "score": float(score)
        })

        seen_titles.add(title)

        if len(results) == top_k:
            break

    return results

In [ ]:
query = "Which movie features the robot Gort?"

results = retrieve_diverse(query, top_k=3, search_k=10)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Title:", result["title"])
    print("Chunk ID:", result["chunk_id"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"][:500], "...")


--- Result 1 ---
Title: The Day the Earth Stood Still
Chunk ID: 2
Score: 0.3943
Text: the spaceship, then leaves to retrieve Klaatu's body. Gort brings Klaatu back to life, but he explains to Helen that his revival is only temporary. Klaatu addresses Barnhardt's assembled scientists, informing them that he represents an interplanetary organization that created a police force of invincible robots like Gort. "In matters of aggression, we have given them absolute power over us". Klaatu concludes, "Your choice is simple: join us and live in peace, or pursue your present course and fa ...

--- Result 2 ---
Title: Yatterman
Chunk ID: 1
Score: 0.3414
Text: trio must find it without losing Shoko. The skull self-destructs. Unknown to the trio, Toybotty witnesses everything and reports back to Gan and Ai. The duo transform into Yatterman and set off for Ogypt. When the group discover the missing piece, the villains arrive on the scene. They again use their mecha to damage Yatterwoof. After cons

In [ ]:
# Inspect similarity scores for the retrieved candidates

for result in results:
    print(
        f"{result['title']}: "
        f"{result['score']:.4f}"
    )

The Day the Earth Stood Still: 0.3943
Yatterman: 0.3414
Kaizoku Sentai Gokaiger vs. Space Sheriff Gavan: The Movie: 0.3388


In [ ]:
def rag_answer(query, top_k=3):
    # Retrieve relevant contexts
    results = retrieve_diverse(
        query=query,
        top_k=top_k,
        search_k=10
    )

    # Build context for the LLM
    context = "\n\n".join(
        [
            f"Movie: {result['title']}\n"
            f"Context: {result['text']}"
            for result in results
        ]
    )

    prompt = f"""
You are a movie plot question-answering assistant.

Answer the user's question using only the retrieved context.

Rules:
1. Do not use outside knowledge.
2. Give a concise and factual answer.
3. If the retrieved context does not contain enough information,
   say that the answer is not available from the retrieved context.
4. Do not invent movie details.

User question:
{query}

Retrieved context:
{context}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return {
        "answer": response.text.strip(),
        "results": results
    }

In [ ]:
result = rag_answer(
    "Which movie features the robot Gort?"
)

print("Answer:")
print(result["answer"])

Answer:
The movie *The Day the Earth Stood Still* features the robot Gort.


In [ ]:
import json

def build_final_output(query, rag_result):
    results = rag_result["results"]

    contexts = [
        {
            "title": result["title"],
            "text": result["text"]
        }
        for result in results
    ]

    reasoning = (
        "The query was converted into an embedding and compared against "
        "the movie-plot embeddings in the FAISS vector store. The most "
        "relevant diverse plot contexts were retrieved and provided to "
        "the LLM, which generated the answer using only those contexts."
    )

    return {
        "answer": rag_result["answer"],
        "contexts": contexts,
        "reasoning": reasoning
    }

In [ ]:
final_result = build_final_output(
    "Which movie features the robot Gort?",
    result
)

print(json.dumps(
    final_result,
    indent=2,
    ensure_ascii=False
))

{
  "answer": "The movie *The Day the Earth Stood Still* features the robot Gort.",
  "contexts": [
    {
      "title": "The Day the Earth Stood Still",
      "text": "the spaceship, then leaves to retrieve Klaatu's body. Gort brings Klaatu back to life, but he explains to Helen that his revival is only temporary. Klaatu addresses Barnhardt's assembled scientists, informing them that he represents an interplanetary organization that created a police force of invincible robots like Gort. \"In matters of aggression, we have given them absolute power over us\". Klaatu concludes, \"Your choice is simple: join us and live in peace, or pursue your present course and face obliteration\". Klaatu and Gort re-enter the spaceship and depart."
    },
    {
      "title": "Yatterman",
      "text": "trio must find it without losing Shoko. The skull self-destructs. Unknown to the trio, Toybotty witnesses everything and reports back to Gan and Ai. The duo transform into Yatterman and set off for Ogy

## 6. Cross-Encoder Reranking

Rerank the retrieved candidate chunks using a cross-encoder to improve relevance and provide higher-quality context to the generation model.

In [ ]:
!pip -q install -U sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded successfully.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker loaded successfully.


In [ ]:
query = "Which movie features the robot Gort?"

# Get more candidates from FAISS
candidate_results = retrieve_diverse(
    query=query,
    top_k=10,
    search_k=20
)

# Create query-document pairs for the reranker
pairs = [
    [query, result["text"]]
    for result in candidate_results
]

# Calculate relevance scores
rerank_scores = reranker.predict(pairs)

# Attach reranking scores
for result, score in zip(candidate_results, rerank_scores):
    result["rerank_score"] = float(score)

# Sort by reranking score
reranked_results = sorted(
    candidate_results,
    key=lambda x: x["rerank_score"],
    reverse=True
)

# Display results
for i, result in enumerate(reranked_results[:5], 1):
    print(f"\n--- Reranked Result {i} ---")
    print("Title:", result["title"])
    print("FAISS score:", round(result["score"], 4))
    print("Rerank score:", round(result["rerank_score"], 4))
    print("Text:", result["text"][:500], "...")


--- Reranked Result 1 ---
Title: The Day the Earth Stood Still
FAISS score: 0.3943
Rerank score: 1.7684
Text: the spaceship, then leaves to retrieve Klaatu's body. Gort brings Klaatu back to life, but he explains to Helen that his revival is only temporary. Klaatu addresses Barnhardt's assembled scientists, informing them that he represents an interplanetary organization that created a police force of invincible robots like Gort. "In matters of aggression, we have given them absolute power over us". Klaatu concludes, "Your choice is simple: join us and live in peace, or pursue your present course and fa ...

--- Reranked Result 2 ---
Title: Kaizoku Sentai Gokaiger vs. Space Sheriff Gavan: The Movie
FAISS score: 0.3388
Rerank score: -5.679
Text: The film begins with the Gokai Galleon being chased by the Super Dimensional Highspeed Ship Dolgiran before crashing into the bay. The Gokaigers (sans Gai, who was sent off to buy dinner for them before this incident) confront the legendary Spa

In [ ]:
# Inspect reranker scores

for result in reranked_results:
    print(
        f"{result['title']}: "
        f"{result['rerank_score']:.4f}"
    )

The Day the Earth Stood Still: 1.7684
Kaizoku Sentai Gokaiger vs. Space Sheriff Gavan: The Movie: -5.6790
Yatterman: -6.0763
Ainthu Ainthu Ainthu: -8.9769
Margam: -9.3256
Cool...Sakkath Hot Maga: -9.3687
Gaily, Gaily: -9.5512
Seeta Rama Jananam: -9.7171
Josettante Hero (ജോസേട്ടന്റെ ഹീറോ): -9.7344
Nick Carter, Master Detective: -10.6484


In [ ]:
def select_relevant_contexts(reranked_results, max_contexts=3):
    return reranked_results[:max_contexts]

In [ ]:
final_contexts = select_relevant_contexts(
    reranked_results,
    max_contexts=3
)

print("Selected contexts:", len(final_contexts))

for i, result in enumerate(final_contexts, 1):
    print(
        f"{i}. {result['title']} "
        f"(rerank score={result['rerank_score']:.4f})"
    )

Selected contexts: 3
1. The Day the Earth Stood Still (rerank score=1.7684)
2. Kaizoku Sentai Gokaiger vs. Space Sheriff Gavan: The Movie (rerank score=-5.6790)
3. Yatterman (rerank score=-6.0763)


In [ ]:
def final_rag_answer(query, max_contexts=3):
    # Step 1: Retrieve candidates from FAISS
    candidate_results = retrieve_diverse(
        query=query,
        top_k=10,
        search_k=20
    )

    # Step 2: Rerank candidates using Cross-Encoder
    pairs = [
        [query, result["text"]]
        for result in candidate_results
    ]

    rerank_scores = reranker.predict(pairs)

    for result, score in zip(candidate_results, rerank_scores):
        result["rerank_score"] = float(score)

    # Step 3: Sort by reranker relevance
    reranked_results = sorted(
        candidate_results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # Step 4: Keep only strongly relevant contexts
    selected_contexts = select_relevant_contexts(
        reranked_results,
        max_contexts=max_contexts
    )

    # Step 5: Build context for Gemini
    context = "\n\n".join(
        [
            f"Movie: {result['title']}\n"
            f"Plot context: {result['text']}"
            for result in selected_contexts
        ]
    )

    # Step 6: Generate answer
    if not selected_contexts:
        answer = (
            "The answer is not available from the retrieved "
            "movie plot context."
        )
    else:
        prompt = f"""
You are a movie plot question-answering assistant.

Answer the user's question using ONLY the retrieved movie plot context.

Rules:
1. Do not use outside knowledge.
2. Give a concise and factual answer.
3. Do not invent movie details.
4. If the context does not contain enough information,
   say that the answer is not available from the retrieved context.

User question:
{query}

Retrieved movie plot context:
{context}
"""

        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        answer = response.text.strip()

    return {
        "answer": answer,
        "contexts": selected_contexts
    }

In [ ]:
test_result = final_rag_answer(
    "Which movie features the robot Gort?"
)

print("Answer:")
print(test_result["answer"])

print("\nContexts:")
for context in test_result["contexts"]:
    print(
        f"- {context['title']} "
        f"(score={context['rerank_score']:.4f})"
    )

Answer:
Based on the retrieved context, the robot Gort is featured in the movie ***The Day the Earth Stood Still***.

Contexts:
- The Day the Earth Stood Still (score=1.7684)
- Kaizoku Sentai Gokaiger vs. Space Sheriff Gavan: The Movie (score=-5.6790)
- Yatterman (score=-6.0763)


In [ ]:
import json

def create_final_json(query, rag_result):
    contexts = [
        result["text"]
        for result in rag_result["contexts"]
    ]

    reasoning = (
        "The query was embedded and searched against the FAISS movie-plot "
        "vector store. The retrieved candidates were reranked using a "
        "cross-encoder, and the top relevant context identified "
        "The Day the Earth Stood Still and its robot Gort. "
        "That context was provided to the LLM to generate the answer."
    )

    return {
        "answer": rag_result["answer"],
        "contexts": contexts,
        "reasoning": reasoning
    }

In [ ]:
final_output = create_final_json(
    "Which movie features the robot Gort?",
    test_result
)

print(json.dumps(
    final_output,
    indent=2,
    ensure_ascii=False
))

{
  "answer": "Based on the retrieved context, the robot Gort is featured in the movie ***The Day the Earth Stood Still***.",
  "contexts": [
    "the spaceship, then leaves to retrieve Klaatu's body. Gort brings Klaatu back to life, but he explains to Helen that his revival is only temporary. Klaatu addresses Barnhardt's assembled scientists, informing them that he represents an interplanetary organization that created a police force of invincible robots like Gort. \"In matters of aggression, we have given them absolute power over us\". Klaatu concludes, \"Your choice is simple: join us and live in peace, or pursue your present course and face obliteration\". Klaatu and Gort re-enter the spaceship and depart.",
    "The film begins with the Gokai Galleon being chased by the Super Dimensional Highspeed Ship Dolgiran before crashing into the bay. The Gokaigers (sans Gai, who was sent off to buy dinner for them before this incident) confront the legendary Space Sheriff Gavan, who quickly

In [ ]:
test_result_2 = final_rag_answer(
    "Which movie involves a sentient dog-shaped mecha?"
)

print("Answer:")
print(test_result_2["answer"])

print("\nContexts:")
for context in test_result_2["contexts"]:
    print(
        f"- {context['title']} "
        f"(score={context['rerank_score']:.4f})"
    )

Answer:
Based on the provided context, the movie that involves a sentient dog-shaped mecha (Yatterwoof) is ***Yatterman***.

Contexts:
- Yatterman (score=4.0848)
- Pudsey: The Movie (score=-8.3300)
- Watchers (score=-8.9368)


In [ ]:
test_result_3 = final_rag_answer(
    "What happens to Klaatu after he is shot?"
)

print("Answer:")
print(test_result_3["answer"])

print("\nContexts:")
for context in test_result_3["contexts"]:
    print(
        f"- {context['title']} "
        f"(score={context['rerank_score']:.4f})"
    )

Answer:
Based on the retrieved context, it is not explicitly mentioned that Klaatu was shot (only that his body is retrieved). 

However, after his body is retrieved:
1. Gort brings Klaatu back to life (though Klaatu explains his revival is only temporary).
2. Klaatu addresses Barnhardt's assembled scientists regarding an interplanetary peace force.
3. Klaatu and Gort re-enter the spaceship and depart.

Contexts:
- The Day the Earth Stood Still (score=-0.2464)
- 1971 (score=-1.4065)
- 8 Thottakkal (score=-3.0696)


In [ ]:
query = "What happens to Klaatu after he is shot?"

candidate_results = retrieve_diverse(
    query=query,
    top_k=10,
    search_k=20
)

pairs = [
    [query, result["text"]]
    for result in candidate_results
]

rerank_scores = reranker.predict(pairs)

for result, score in zip(candidate_results, rerank_scores):
    result["rerank_score"] = float(score)

reranked_test = sorted(
    candidate_results,
    key=lambda x: x["rerank_score"],
    reverse=True
)

for i, result in enumerate(reranked_test, 1):
    print(
        f"{i}. {result['title']} "
        f"| score={result['rerank_score']:.4f}"
    )

1. The Day the Earth Stood Still | score=-0.2464
2. 1971 | score=-1.4065
3. 8 Thottakkal | score=-3.0696
4. After Tonight | score=-3.2274
5. Kalidasu | score=-3.2959
6. Kamen Rider Kabuto: GOD SPEED LOVE | score=-6.9413
7. Anjathe | score=-7.2921
8. Assassination Classroom: Graduation | score=-7.3806
9. Tale of Zatoichi Continues !The Tale of Zatoichi Continues | score=-8.8558
10. Inaam Dus Hazaar | score=-9.2279


In [ ]:
test_result_3 = final_rag_answer(
    "What happens to Klaatu after he is shot?"
)

print("Answer:")
print(test_result_3["answer"])

print("\nContexts:")
for context in test_result_3["contexts"]:
    print(
        f"- {context['title']} "
        f"(score={context['rerank_score']:.4f})"
    )

Answer:
Based on the provided context, Gort retrieves Klaatu's body and brings him back to life, though Klaatu explains to Helen that his revival is only temporary. Klaatu then addresses Barnhardt's assembled scientists to give them a warning, after which he and Gort re-enter the spaceship and depart.

Contexts:
- The Day the Earth Stood Still (score=-0.2464)
- 1971 (score=-1.4065)
- 8 Thottakkal (score=-3.0696)


In [ ]:
def retrieve_candidates(query, search_k=20):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "title": chunks_df.iloc[idx]["title"],
            "chunk_id": int(chunks_df.iloc[idx]["chunk_id"]),
            "text": chunks_df.iloc[idx]["text"],
            "score": float(score)
        })

    return results

In [ ]:
query = "What happens to Klaatu after he is shot?"

candidate_results = retrieve_candidates(
    query=query,
    search_k=20
)

print("Number of candidates:", len(candidate_results))

for i, result in enumerate(candidate_results[:10], 1):
    print(f"\n--- Candidate {i} ---")
    print("Title:", result["title"])
    print("Chunk ID:", result["chunk_id"])
    print("FAISS score:", round(result["score"], 4))
    print("Text:", result["text"][:300], "...")

In [ ]:
query = "What happens to Klaatu after he is shot?"

candidate_results = retrieve_candidates(
    query=query,
    search_k=100
)

print("Number of candidates:", len(candidate_results))

# Show all chunks belonging to The Day the Earth Stood Still
for result in candidate_results:
    if result["title"] == "The Day the Earth Stood Still":
        print(
            f"Chunk ID: {result['chunk_id']} "
            f"| FAISS score: {result['score']:.4f}"
        )
        print(result["text"][:500])
        print("-" * 80)

In [ ]:
query = "What happens to Klaatu after he is shot?"

# Retrieve 100 candidates
candidate_results = retrieve_candidates(
    query=query,
    search_k=100
)

# Rerank all candidates
pairs = [
    [query, result["text"]]
    for result in candidate_results
]

rerank_scores = reranker.predict(pairs)

for result, score in zip(candidate_results, rerank_scores):
    result["rerank_score"] = float(score)

# Sort by reranker score
reranked_klaatu = sorted(
    candidate_results,
    key=lambda x: x["rerank_score"],
    reverse=True
)

# Show top 10
for i, result in enumerate(reranked_klaatu[:10], 1):
    print(
        f"{i}. {result['title']} "
        f"| Chunk {result['chunk_id']} "
        f"| Rerank score={result['rerank_score']:.4f}"
    )

In [ ]:
def final_rag_answer_v2(query, max_contexts=3):
    # 1. Retrieve broad candidate set
    candidate_results = retrieve_candidates(
        query=query,
        search_k=100
    )

    # 2. Rerank candidates
    pairs = [
        [query, result["text"]]
        for result in candidate_results
    ]

    rerank_scores = reranker.predict(pairs)

    for result, score in zip(candidate_results, rerank_scores):
        result["rerank_score"] = float(score)

    # 3. Sort by relevance
    reranked_results = sorted(
        candidate_results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # 4. Select top relevant contexts
    selected_contexts = reranked_results[:max_contexts]

    # 5. Build LLM context
    context = "\n\n".join(
        [
            f"Movie: {result['title']}\n"
            f"Plot context: {result['text']}"
            for result in selected_contexts
        ]
    )

    # 6. Generate answer
    prompt = f"""
You are a movie plot question-answering assistant.

Answer the user's question using ONLY the retrieved movie plot context.

Rules:
1. Do not use outside knowledge.
2. Give a concise and factual answer.
3. Do not invent movie details.
4. If the context does not contain enough information,
   say that the answer is not available from the retrieved context.

User question:
{query}

Retrieved movie plot context:
{context}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return {
        "answer": response.text.strip(),
        "contexts": selected_contexts
    }

In [ ]:
test_result_3_v2 = final_rag_answer_v2(
    "What happens to Klaatu after he is shot?"
)

print("Answer:")
print(test_result_3_v2["answer"])

print("\nContexts:")

for i, context in enumerate(test_result_3_v2["contexts"], 1):
    print(
        f"{i}. {context['title']} "
        f"| Chunk {context['chunk_id']} "
        f"| score={context['rerank_score']:.4f}"
    )

In [ ]:
test_result_gort_v2 = final_rag_answer_v2(
    "Which movie features the robot Gort?"
)

print("Answer:")
print(test_result_gort_v2["answer"])

print("\nContexts:")

for i, context in enumerate(test_result_gort_v2["contexts"], 1):
    print(
        f"{i}. {context['title']} "
        f"| Chunk {context['chunk_id']} "
        f"| score={context['rerank_score']:.4f}"
    )

In [ ]:
unknown_test = final_rag_answer_v2(
    "Who directed Titanic?"
)

print("Answer:")
print(unknown_test["answer"])

print("\nContexts:")

for i, context in enumerate(unknown_test["contexts"], 1):
    print(
        f"{i}. {context['title']} "
        f"| Chunk {context['chunk_id']} "
        f"| score={context['rerank_score']:.4f}"
    )

## 8. Structured JSON Output

Return the generated answer together with the retrieved contexts and a concise reasoning summary in a structured JSON format.

In [ ]:
import json

def final_json_answer(query, max_contexts=3):
    rag_result = final_rag_answer_v2(
        query=query,
        max_contexts=max_contexts
    )

    contexts = [
        result["text"]
        for result in rag_result["contexts"]
    ]

    if contexts:
        reasoning = (
            "The query was embedded and searched against the FAISS "
            "movie-plot vector store using a broad candidate search. "
            "The retrieved candidates were reranked using a "
            "cross-encoder, and the highest-ranked contexts were "
            "provided to the LLM to generate a grounded answer."
        )
    else:
        reasoning = (
            "No sufficiently relevant movie-plot context was retrieved, "
            "so the system did not generate an unsupported answer."
        )

    return {
        "answer": rag_result["answer"],
        "contexts": contexts,
        "reasoning": reasoning
    }

## 9. Evaluation & Test Cases

Validate the RAG pipeline using factual, multi-context, and out-of-scope questions to assess retrieval quality, grounded answer generation, and resistance to unsupported answers.

In [ ]:
final_output = final_json_answer(
    "Which movie features the robot Gort?"
)

print(json.dumps(
    final_output,
    indent=2,
    ensure_ascii=False
))

In [ ]:
# Evaluation Test 1: Robot Gort

test_queries = [
    "Which movie features the robot Gort?",
    "Which movie involves a sentient dog-shaped mecha?",
    "What happens to Klaatu after he is shot?",
    "Who directed Titanic?"
]

for i, query in enumerate(test_queries, 1):
    result = final_json_answer(query)

    print(f"\n{'=' * 60}")
    print(f"TEST {i}")
    print(f"Question: {query}")
    print(f"Answer: {result['answer']}")
    print("Retrieved contexts:")

    for j, context in enumerate(result["contexts"], 1):
        print(f"  {j}. {context[:150]}...")

### Evaluation Summary

The RAG pipeline was evaluated using factual, multi-context, and out-of-scope questions.

- **Gort retrieval:** Correctly identified *The Day the Earth Stood Still*.
- **Yatterwoof retrieval:** Correctly identified *Yatterman*.
- **Klaatu reasoning:** Correctly combined multiple retrieved plot chunks to answer the question.
- **Out-of-scope query:** The system declined to answer when the required information was not available in the retrieved context.

These tests demonstrate semantic retrieval, cross-encoder reranking, grounded generation, and resistance to unsupported answers.